# 🏗️ Notebook: LLM Parameters & Structured Outputs

Before we force an LLM into a fixed output format, we look at the knobs that control *how* it generates text. Then we turn free text into data we can work with in code, e.g. a dict with fixed fields:

1. **LLM inference parameters:** context size, temperature, top-k/top-p, max tokens, stop sequences
2. **Pydantic basics:** describe the shape of your data as a Python class
3. **Structured outputs:** force the LLM to answer in exactly that shape

## 📚 Sources

- [OpenAI API Reference: Chat Completions](https://platform.openai.com/docs/api-reference/chat/create)
- [Ollama API: Model Parameters](https://github.com/ollama/ollama/blob/main/docs/modelfile.md#valid-parameters-and-values)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [OpenAI: Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat)
- [Ollama Structured Outputs](https://ollama.com/blog/structured-outputs)

# Part 1: LLM Inference Parameters

Inference parameters are settings that control **how** the model answers, without changing the prompt:

| # | Parameter | Question it answers | In the API |
|---|-----------|---------------------|------------|
| 1 | Context size | How much can the model read? | fixed by model/server |
| 2 | Temperature | How creative should it be? | `temperature` |
| 3 | Top-k & Top-p | Which tokens are allowed? | `top_p` (`top_k` not in the OpenAI API) |
| 4 | Max tokens | How long may the answer be? | `max_tokens` |
| 5 | Stop sequence | When does it stop? | `stop` |

First we connect to the course LLM server, just like in notebook 03.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads LLM_HOST, LLM_PORT and LLM_API_KEY from the .env file (see setup.md)

LLM_HOST = os.environ["LLM_HOST"]
LLM_PORT = os.environ["LLM_PORT"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_URL = f"http://{LLM_HOST}:{LLM_PORT}/v1"
LLM_REASONING = "gemma4:26b"

client = OpenAI(base_url=LLM_URL, api_key=LLM_API_KEY)

## 1. Context Size

The context window is the maximum number of tokens the model can process at once: **prompt + answer**. It is set by the model and the server, not per request in the OpenAI API. What we *can* see is how many tokens a request used, via `completion.usage`:

In [ ]:
completion = client.chat.completions.create(
    model=LLM_REASONING,
    reasoning_effort="none",
    messages=[{"role": "user", "content": "What is the capital of France? Answer in one sentence."}],
)

print(completion.choices[0].message.content)
print(completion.usage)

## 2. Temperature

The logits are divided by `temperature` before the softmax. **Low** values sharpen the distribution (the top token almost always wins), **high** values flatten it (rare tokens get picked more often).

We ask the same question three times with `temperature=0` ...

In [ ]:
prompt = "Invent a name for a new café in Regensburg. Answer with the name only."

for _ in range(3):
    completion = client.chat.completions.create(
        model=LLM_REASONING,
        reasoning_effort="none",
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    print(completion.choices[0].message.content)

... and three times with `temperature=1.5`:

In [ ]:
for _ in range(3):
    completion = client.chat.completions.create(
        model=LLM_REASONING,
        reasoning_effort="none",
        temperature=1.5,
        messages=[{"role": "user", "content": prompt}],
    )
    print(completion.choices[0].message.content)

## 3. Top-k & Top-p

Both restrict *which* tokens can be sampled at all:

- **Top-k:** only the `k` most likely tokens.
- **Top-p:** the smallest set of tokens whose probabilities add up to at least `p` (adaptive size).

With `top_p=0.1` only the very top tokens remain, so even with `temperature=1.5` we get (almost) the same answer every time:

In [ ]:
for _ in range(3):
    completion = client.chat.completions.create(
        model=LLM_REASONING,
        reasoning_effort="none",
        temperature=1.5,
        top_p=0.1,
        messages=[{"role": "user", "content": prompt}],
    )
    print(completion.choices[0].message.content)

⚠️ `top_k` is **not** a parameter of the OpenAI API, so the `openai` package has no `top_k` argument. Other libraries (e.g. Ollama's native API, vLLM, Hugging Face `transformers`) do support it.

## 4. Max Tokens

`max_tokens` caps the number of **generated** tokens. If the limit is hit, the answer is simply cut off, and `finish_reason` is `"length"` instead of `"stop"`:

In [ ]:
completion = client.chat.completions.create(
    model=LLM_REASONING,
    reasoning_effort="none",
    max_tokens=10,
    messages=[{"role": "user", "content": "Explain photosynthesis."}],
)

print(completion.choices[0].message.content)
print(completion.choices[0].finish_reason)

## 5. Stop Sequence

`stop` takes one or more strings. Generation ends as soon as the model produces one of them; the stop sequence itself is not included in the answer. Here: stop at the first line break.

In [ ]:
completion = client.chat.completions.create(
    model=LLM_REASONING,
    reasoning_effort="none",
    stop=["\n"],
    messages=[{"role": "user", "content": "List three fruits, one per line."}],
)

print(repr(completion.choices[0].message.content))

# Part 2: Pydantic Basics

## 1. A Basic Model

A Pydantic model is a class that inherits from `BaseModel`. Instead of writing `__init__`, you simply list the attributes with their types. Pydantic generates the constructor and validation for you.

In [ ]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

person = Person(name="Alice", age=30)
print(person)
print(person.name, person.age)

## 2. Validation

Compatible values are converted automatically (`"25"` → `25`). Values that don't fit raise a `ValidationError`. This is the whole point: bad data is caught instead of silently accepted.

In [ ]:
from pydantic import ValidationError

print(Person(name="Bob", age="25"))  # "25" is converted to int

try:
    Person(name="Charlie", age="not a number")
except ValidationError as e:
    print(e)

## 3. `Field()`, `Literal` and Optional Fields

- `Field()` adds constraints, e.g. `gt`/`ge`/`lt`/`le` (greater/less than (or equal)) or `min_length`/`max_length`.
- `Literal[...]` allows only a fixed set of values, perfect for categories.
- `X | None = None` makes a field optional (default `None`).

In [ ]:
from typing import Literal
from pydantic import Field

class Product(BaseModel):
    name: str
    price: float = Field(gt=0)                       # must be > 0
    category: Literal["food", "drinks", "other"]     # only these three values
    note: str | None = None                          # optional

print(Product(name="Coffee", price=2.5, category="drinks"))

try:
    Product(name="Coffee", price=-1, category="clothes")
except ValidationError as e:
    print(e)

## 4. Nested Models

Models can contain other models, also as lists. Pydantic validates the whole structure.

In [ ]:
class Address(BaseModel):
    city: str
    zip_code: str

class Customer(BaseModel):
    name: str
    addresses: list[Address]

customer = Customer(name="Frank", addresses=[{"city": "Munich", "zip_code": "80331"}])
print(customer.addresses[0].city)

## 5. JSON In, JSON Out

Two methods connect Pydantic to LLMs:

- `model_validate_json()` parses a JSON string into a validated object. Remember: an LLM only ever generates text. Even when it "returns JSON", it is just a string that looks like JSON. With this method we turn that string into a real Python object.
- `model_json_schema()` turns the model into a **JSON Schema**, a standard description of the expected structure. This schema is what we send to the LLM.

In [ ]:
import json

person = Person.model_validate_json('{"name": "Grace", "age": 28}')
print(person)
print(person.model_dump())  # back to a dict

print(json.dumps(Person.model_json_schema(), indent=2))

# Part 3: Structured Outputs with LLMs

**Structured outputs** restrict the model's output to a given JSON schema. We pass a Pydantic model as `response_format` and the server (here Ollama via its OpenAI-compatible API) makes sure the answer matches the schema.

We reuse the `client` from Part 1.

Two habits for the examples below:

- **Include the schema in the prompt.** `response_format` only enforces the *shape*. The model still needs to know what each field *means*, so we also put the schema into the prompt.
- **Disable thinking for simple tasks** with `reasoning_effort="none"`. It saves time without hurting quality here.

## Example 1: Classification

`SentimentResult.model_json_schema()` turns the Pydantic class into a **JSON Schema**: a Python dict that describes every field with its name, type and allowed values (here: `sentiment` is a string, one of `"positive"`, `"neutral"`, `"negative"`). `json.dumps(..., indent=2)` turns this dict into a readable string, so we can put it into the prompt. That way the model can actually read which fields it should fill and what they mean.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]

prompt = f"""Classify the sentiment of this text: "I love programming in Python!"

Answer as JSON matching this schema:
{json.dumps(SentimentResult.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=SentimentResult,
)

result = completion.choices[0].message.parsed  # already a SentimentResult object
print(result)
print(result.sentiment)

`.message.content` contains the raw JSON string, `.message.parsed` the validated Pydantic object.

## Example 2: Extracting Information from Text

The same pattern works for nested structures, e.g. a list of courses. Note that `semester` is optional: the second course has no semester in the text.

In [ ]:
class Course(BaseModel):
    title: str
    professor: str
    credits: int
    semester: str | None = None

class CourseList(BaseModel):
    courses: list[Course]

text = """I'm enrolled in two courses. "Introduction to Machine Learning" is taught by Prof. Müller,
worth 6 credits and offered in the winter semester. I'm also taking "Database Systems",
a 5-credit course taught by Prof. Schmidt."""

prompt = f"""Extract all courses from this text:
{text}

Answer as JSON matching this schema:
{json.dumps(CourseList.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=CourseList,
)

for course in completion.choices[0].message.parsed.courses:
    print(course)

### Exercise 1

Let the LLM generate dummy data for **2 researchers**.

| Field          | Description                                                      |
| -------------- | ---------------------------------------------------------------- |
| `name`         | Name of the researcher                                           |
| `age`          | Age between 25 and 80                                            |
| `field`        | One of: Computer Science, Biology, Physics, Mathematics          |
| `publications` | List of publications, each with `title` and `year` (1900–2025)   |

💡 Use `Field()` and `Literal`, and include the JSON schema in the prompt.

<details>
<summary><b>Show Solution</b></summary>

```python
class Publication(BaseModel):
    title: str
    year: int = Field(ge=1900, le=2025)

class Researcher(BaseModel):
    name: str
    age: int = Field(ge=25, le=80)
    field: Literal["Computer Science", "Biology", "Physics", "Mathematics"]
    publications: list[Publication]

class ResearcherList(BaseModel):
    researchers: list[Researcher] = Field(min_length=2, max_length=2)

prompt = f"""Generate 2 fictional researchers.

Answer as JSON matching this schema:
{json.dumps(ResearcherList.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=ResearcherList,
)

for researcher in completion.choices[0].message.parsed.researchers:
    print(researcher)
```

</details>

In [ ]:
# Your code goes here...

## ADVANCED – Example 3: Regex Instead of JSON

*Optional: this goes beyond the standard JSON workflow and is specific to Ollama.*

Ollama's native API (`/api/chat`, not the OpenAI-compatible `/v1`) can also constrain the output to a **regex pattern**. The `openai` package silently ignores this, so we use `requests` here.

⚠️ Keep patterns tight, e.g. `[A-Z][a-z]+` or `[0-9]{1,3}`. An open `.*` can make Ollama hang for minutes, and loose patterns like `[A-Za-z ]+` let the model add stray spaces.

Since the format is technically a JSON schema of type `string`, Ollama returns a JSON string **including the quotes** (`"Name: ..."`). `json.loads()` turns it into a normal Python string.

In [ ]:
import requests

data = {
    "model": LLM_REASONING,
    "messages": [{"role": "user", "content": "Create a name and age for a random person."}],
    "stream": False,
    "think": False,
    "format": {"type": "string", "pattern": "^Name: [A-Z][a-z]+ [A-Z][a-z]+, Age: [0-9]{1,3}$"},
}

response = requests.post(
    f"http://{LLM_HOST}:{LLM_PORT}/api/chat",
    json=data,
    headers={"Authorization": f"Bearer {LLM_API_KEY}"},
)
raw = response.json()["message"]["content"]
print(raw)              # JSON string with quotes
print(json.loads(raw))  # plain Python string

### Exercise 2

Extract a structured order from this customer email:

> Hi, I'd like to order 3 notebooks at 2.50 EUR each and 1 backpack for 39.90 EUR. Please ship it to Regensburg, 93053. Thanks, Lena Weber

The result should contain the customer name, the shipping address (city, zip code) and a list of items (name, quantity, unit price).

<details>
<summary><b>Show Solution</b></summary>

```python
class Item(BaseModel):
    name: str
    quantity: int = Field(ge=1)
    unit_price_eur: float = Field(gt=0)

class Order(BaseModel):
    customer_name: str
    shipping_address: Address  # reuses the Address model from Part 2
    items: list[Item]

email = """Hi, I'd like to order 3 notebooks at 2.50 EUR each and 1 backpack for 39.90 EUR.
Please ship it to Regensburg, 93053. Thanks, Lena Weber"""

prompt = f"""Extract the order from this email:
{email}

Answer as JSON matching this schema:
{json.dumps(Order.model_json_schema(), indent=2)}
"""

completion = client.chat.completions.parse(
    model=LLM_REASONING,
    temperature=0,
    reasoning_effort="none",
    messages=[{"role": "user", "content": prompt}],
    response_format=Order,
)

order = completion.choices[0].message.parsed
print(order)
print("Total:", sum(item.quantity * item.unit_price_eur for item in order.items))
```

</details>

In [ ]:
# Your code goes here...